# COMP7103 Lab 5 · Test an assistant's model suggestion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab5/Lab5_model_check.ipynb)

300 past loan applications, 20 columns, 30% bad credit. You ask a chat LLM which model to use, then **test its suggestion like any other model**: the same data, the same 5 folds, against the classical models you know.

| Time | What you do |
|---|---|
| 5 min | Brief: open the notebook and run Setup |
| 30 min | Guided work: ask your chat LLM, baselines and its model, the deep net, then all 1,000 rows |
| 10 min | Share-out |

This lab is ungraded practice for the exams. Data: German Credit (UCI Statlog), CC BY 4.0.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the 300-row sample and all 1,000 rows. The target is `bad_credit` (1 = bad).

In [ ]:
import os, time, warnings, urllib.request
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab5/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 5 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 5 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab5")
    return os.path.join("data", name)

df300 = pd.read_csv(get("credit_300.csv"))
df1000 = pd.read_csv(get("credit_1000.csv"))
# X: the 20 attribute columns; y: the target column (1 = bad credit, 0 = good)
X300 = df300.drop(columns="bad_credit")
y300 = df300["bad_credit"]
X1000 = df1000.drop(columns="bad_credit")
y1000 = df1000["bad_credit"]
print("300-row sample:", X300.shape, "· all rows:", X1000.shape)

The helpers (given). `score()` runs one model on the same 5 folds and returns its mean ROC-AUC (area under the ROC curve, Lecture 4: 0.5 = coin flip, 1 = perfect ranking), the spread of the ROC-AUC over the folds (`± sd`, the standard deviation) and the mean fit time; `compare()` does it for a whole dictionary of models.

In [ ]:
FOLDS = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # the same 5 folds for every model

def prep():
    """One-hot encode the 13 category columns and standardise the 7 number columns."""
    category_columns = X1000.select_dtypes(include="object").columns    # text columns, e.g. purpose
    number_columns = X1000.select_dtypes(exclude="object").columns      # number columns, e.g. age
    return make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore"), category_columns),      # one 0/1 column per category
        (StandardScaler(), number_columns))                              # z-scores (Lecture 3)

def score(model, X, y):
    """Run prep + model on the same 5 folds. Returns the mean ROC-AUC, its standard deviation and the mean fit time."""
    pipeline = make_pipeline(prep(), model)            # first prep(), then the model
    cv = cross_validate(pipeline, X, y, cv=FOLDS, scoring="roc_auc")
    auc_per_fold = cv["test_score"]                    # 5 numbers, one per test fold
    seconds_per_fold = cv["fit_time"]
    return {"ROC-AUC": round(auc_per_fold.mean(), 3),
            "± sd": round(auc_per_fold.std(), 3),
            "fit time (s)": round(seconds_per_fold.mean(), 3)}

def compare(models, X, y):
    """Score every model in the dictionary {name: model} on the same data and folds. Returns one row per model."""
    table = {}
    for name in models:
        table[name] = score(models[name], X, y)
    return pd.DataFrame(table).T                       # .T turns the table so that each model is a row

## 1 · Ask your chat LLM (5 min)

1. Run the next cell. It prints a message that describes the table.
2. Paste the message into any chat LLM you use. The message holds no personal data.
3. Paste the LLM's whole answer between the triple quotes in the cell after, and write down the model it suggests. If the cell then stops with a `SyntaxError`, delete the code part of the answer (code often holds triple quotes or backslashes).

No LLM available? Use the ready-made answer (see the comment in that cell). It comes from Qwen2.5-1.5B-Instruct, a small open model.

In [ ]:
print(open(get("llm_prompt.txt")).read())

In [ ]:
llm_reply = """
paste the LLM's answer here
"""
# No LLM? Use the ready-made answer instead:
# llm_reply = open(get("llm_fallback.txt")).read()
print(llm_reply)
SUGGESTED = "write the model it suggests here"

## 2 · Look at the data (2 min)

In [ ]:
df300.head()

In [ ]:
print("share of bad credit:", round(y300.mean(), 2))
X300.dtypes.value_counts()

## 3 · The models you know, and your LLM's model (about 9 min)

Fill in the blanks: a majority baseline (ZeroR, Lecture 4: it always predicts the larger class), a decision tree of depth 4, logistic regression and a random forest of 300 trees. Then score all four on the same folds.

In [ ]:
MODELS = {
    "Majority baseline": DummyClassifier(strategy=___),     # a string: sklearn's name for "the most frequent class"
    "Decision tree": DecisionTreeClassifier(max_depth=___, random_state=0),
    "Logistic regression": ___(max_iter=1000),
    "Random forest": RandomForestClassifier(n_estimators=___, random_state=0),
}
results300 = compare(MODELS, X300, y300)
results300

ROC-AUC 0.5 is a coin flip (the majority baseline); 1.0 is perfect.

Now test your LLM's suggestion like the others: add the model it named, **with the settings it named**, to the dictionary. If it named a model that is not imported here (for example XGBoost), use the closest one: `GradientBoostingClassifier`. If it suggested the deep net of Section 4, enter it here with its settings anyway.

In [ ]:
# Examples (not your answer):
#   RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0)
#   GradientBoostingClassifier(random_state=0)
#   SVC(kernel="rbf", C=1.0)
#   LogisticRegression(C=0.5, max_iter=1000)
MODELS["LLM suggestion"] = ___
results300 = compare(MODELS, X300, y300)
results300

## 4 · The deep net an assistant often suggests (about 8 min)

A deep net is a stack of layers of units; each unit is like a small logistic regression on the outputs of the layer below (Lecture 6 covers it). Here: three hidden layers of 128, 64 and 32 units and the Adam optimizer (a training method; Lecture 6). Add it to the dictionary and score it **on the same folds**.

In [ ]:
MODELS["Deep net (128-64-32)"] = MLPClassifier(hidden_layer_sizes=(___, ___, ___), max_iter=300,
                                               random_state=0)
results300 = compare(MODELS, X300, y300)
results300

How many numbers does the deep net learn? Fit it once on the 300 rows and count its weights (`coefs_` holds the weight matrices, `intercepts_` the bias vectors).

In [ ]:
pipeline = make_pipeline(prep(), MODELS["Deep net (128-64-32)"])
pipeline.fit(X300, y300)
net = pipeline[-1]                     # the last step of the pipeline: the trained deep net

n_weights = 0
for w in net.coefs_:                   # one weight matrix per layer
    n_weights = n_weights + w.size     # .size = number of entries in the matrix
for b in net.___:                      # one bias vector per layer
    n_weights = n_weights + b.size
print(f"The deep net learns {n_weights:,} weights from {len(y300)} rows.")    # :, writes 18305 as 18,305

**Is a gap real?** Compare it with the `± sd` column: at 300 rows the ROC-AUC moves by 0.07 to 0.12 from fold to fold, so a gap under 0.01 is a tie. The lecture's chart averages 10 different 300-row samples, so its numbers differ slightly from yours (there: deep net 0.74, logistic regression and random forest 0.75).

## 5 · All 1,000 rows, then your verdict (about 6 min)

One line changes: the data. Same models, same kind of folds.

In [ ]:
results1000 = compare(MODELS, ___, ___)
# Put the two tables side by side: 300 rows on the left, 1,000 rows on the right
both = pd.concat({"300 rows": results300, "1,000 rows": results1000}, axis=1)
both

The assistant's settings were never tuned. A deep net has a setting called **weight decay** (`alpha`): a penalty that keeps its 18,305 weights small. Let cross-validation *inside each training fold* choose it from 0.0001, 1 and 10, then score the tuned net on all 1,000 rows, on the same outer folds, and add it to the 1,000-row table. This cell takes about 45 seconds (up to a minute on Colab).

In [ ]:
from sklearn.model_selection import GridSearchCV
net = MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=300, random_state=0)
alphas_to_try = {"alpha": [0.0001, 1, 10]}
# GridSearchCV tries each alpha with 3-fold cross-validation inside the training data, and keeps the best one
tuned_net = GridSearchCV(net, alphas_to_try, cv=3, scoring="roc_auc")
tuned_scores = score(tuned_net, ___, ___)
results1000.loc["Deep net, tuned"] = pd.Series(tuned_scores)     # add one row to the table
results1000

**Your verdict on the deep net for this 300-row table.** Accept it, test it further first, or reject it? Write one sentence why, with numbers from the 300-row columns. Then say whether the 1,000-row columns change your mind.

In [ ]:
VERDICT = "accept / test first / reject"   # the deep net, for the 300-row table
WHY = "one sentence, with numbers from the 300-row columns; then: do the 1,000 rows change your mind?"
print(VERDICT, "·", WHY)

## 6 · Share-out (10 min)

- Report your ROC-AUC at 300 rows for logistic regression, the random forest and the deep net, and your verdict on the deep net for the 300-row table (accept / test first / reject); we collect them on the class board.
- What did your chat LLM suggest, and how did its model score against the others? Did it ask how much data you had, or mention a baseline?
- The habit from the lecture: a suggested model, from an AI or a person, is tested like any other model, on the same data and folds.